In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/genz_mental_wellness_synthetic_dataset.csv")

In [3]:
print(df.shape)
df.head()

(10000, 22)


,Age,Gender,Country,Student_Working_Status,Daily_Social_Media_Hours,Screen_Time_Hours,Night_Scrolling_Frequency,Online_Gaming_Hours,Content_Type_Preference,Exercise_Frequency_per_Week,...,Study_Work_Hours_per_Day,Overthinking_Score,Anxiety_Score,Mood_Stability_Score,Social_Comparison_Index,Sleep_Quality_Score,Motivation_Level,Emotional_Fatigue_Score,Wellbeing_Index,Burnout_Risk
0,24,Male,Canada,Working,4.81,6.93,2.61,2.07,News,5.41,...,11.42,4.95,4.13,5.74,4.67,6.27,6.13,6.45,4.28,Medium
1,21,Male,USA,Student,4.16,7.94,1.85,3.58,Gaming,3.41,...,6.98,5.91,3.63,5.75,5.38,7.37,6.27,3.74,5.23,Medium
2,25,Male,Pakistan,Student,3.07,7.45,2.96,2.85,Entertainment,3.40,...,7.79,4.06,5.67,6.03,2.41,6.48,4.82,6.69,3.72,High
3,22,Female,Pakistan,Student,4.41,7.34,4.51,3.37,Educational,2.19,...,6.61,6.10,4.78,4.85,5.86,7.27,5.17,5.96,3.97,High
4,24,Male,Pakistan,Student,2.97,5.76,2.36,1.77,Educational,4.93,...,4.81,5.22,4.23,5.05,5.54,6.34,5.72,2.22,4.63,Medium


In [4]:
X_burnout = df.drop(
    ["Burnout_Risk", "Wellbeing_Index"],
    axis=1
)

y_burnout = df["Burnout_Risk"]

In [5]:
X_wellbeing = df.drop(
    ["Wellbeing_Index", "Burnout_Risk"],
    axis=1
)

y_wellbeing = df["Wellbeing_Index"]

In [6]:
cat_cols = [
    "Gender",
    "Country",
    "Student_Working_Status",
    "Content_Type_Preference"
]

num_cols = [
    col for col in X_burnout.columns
    if col not in cat_cols
]

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [8]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            num_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            cat_cols
        )
    ]
)

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

In [10]:
burnout_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced"
        )
    )
])

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X_burnout,
    y_burnout,
    test_size=0.2,
    random_state=42,
    stratify=y_burnout
)

In [13]:
burnout_pipeline.fit(
    X_train,
    y_train
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [14]:
from sklearn.metrics import classification_report

In [15]:
preds = burnout_pipeline.predict(X_test)

print(
    classification_report(
        y_test,
        preds
    )
)

              precision    recall  f1-score   support

        High       0.96      0.97      0.96      1078
         Low       1.00      0.15      0.27        13
      Medium       0.95      0.95      0.95       909

    accuracy                           0.96      2000
   macro avg       0.97      0.69      0.73      2000
weighted avg       0.96      0.96      0.95      2000



In [16]:
from sklearn.ensemble import RandomForestRegressor

In [17]:
wellbeing_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42
        )
    )
])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X_wellbeing,
    y_wellbeing,
    test_size=0.2,
    random_state=42
)

In [21]:
wellbeing_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [22]:
preds = wellbeing_pipeline.predict(X_test)

In [18]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

In [23]:


print(
    "MAE:",
    mean_absolute_error(y_test, preds)
)

print(
    "R2:",
    r2_score(y_test, preds)
)

MAE: 0.10273050000000009
R2: 0.9856794786609849


In [24]:
from sklearn.inspection import permutation_importance

In [26]:
result = permutation_importance(
    wellbeing_pipeline,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42
)

feature_names = wellbeing_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
})

print(
    importance_df.sort_values(
        "importance",
        ascending=False
    ).head(15)
)

                        feature  importance
17          Sleep_Quality_Score    0.277724
18             Motivation_Level    0.233390
15         Mood_Stability_Score    0.196800
14                Anxiety_Score    0.111141
19      Emotional_Fatigue_Score    0.043757
10            Daily_Sleep_Hours    0.001086
9   Exercise_Frequency_per_Week    0.000179
13           Overthinking_Score    0.000105
5             Screen_Time_Hours    0.000088
4      Daily_Social_Media_Hours    0.000077
16      Social_Comparison_Index    0.000030
12     Study_Work_Hours_per_Day    0.000025
2                       Country    0.000018
1                        Gender    0.000016
7           Online_Gaming_Hours    0.000013


In [27]:
import joblib

In [28]:
joblib.dump(
    burnout_pipeline,
    "../models/burnout_model.pkl"
)

joblib.dump(
    wellbeing_pipeline,
    "../models/wellbeing_model.pkl"
)

['../models/wellbeing_model.pkl']

In [29]:
import os

print(os.listdir("../models"))

['burnout_model.pkl', 'wellbeing_model.pkl']


In [30]:
loaded_burnout = joblib.load(
    "../models/burnout_model.pkl"
)

loaded_wellbeing = joblib.load(
    "../models/wellbeing_model.pkl"
)

In [31]:
loaded_burnout.predict(
    X_test.iloc[:5]
)

array(['High', 'High', 'High', 'Medium', 'High'], dtype=object)

In [32]:
loaded_wellbeing.predict(
    X_test.iloc[:5]
)

array([3.7143    , 3.73343333, 2.25896667, 5.91593333, 2.73703333])